# Bayesian-Value Fine-Tuning

Fine-tunes the Bayesian-Value model (softmax best response every stage)
on both aggregate and switch-stage metrics.

In [1]:
import sys
import json
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from itertools import product
from scipy.optimize import minimize

# Shared package
from shared import EXPORTS_DIR, DATA_ROOT
from shared.constants import (
    F, T, M,
    ROLE_NAMES, ROLE_SHORT, ROLE_CHAR_TO_IDX, GAME_ROLE_TO_IDX,
    ALL_ROLE_COMBOS, EPSILON, MAX_STAGES, TURNS_PER_STAGE,
)
from shared.parsing import canonical_combo, get_canonical_combos
from shared.inference import (
    utility_based_prior, uniform_prior,
    bayesian_update, action_prob, preferred_action, game_step,
    softmax_role_dist, combo_marginal,
    ATTACK, DEFEND, HEAL,
)
from shared.evaluation import (
    run_predictions, compute_pearson, compute_log_likelihood, extract_metrics,
)

# Still need online_model_sim for load_team_rounds
OMS_DIR = Path(DATA_ROOT).parent.parent / 'computational_model' / 'analysis'
sys.path.insert(0, str(OMS_DIR))
import online_model_sim as oms

## 1. Load Data

In [2]:
# Monkey-patch env paths to use shared data directory
oms.VALUE_MATRICES_DIR = DATA_ROOT / 'human_envs_value_matrices'
oms.ENVS_DIR = DATA_ROOT / 'envs'

# Monkey-patch config loader to avoid JAX dependency
from shared.env_loading import load_env_config
import re as _re

def _load_config_no_jax(config_path):
    text = Path(config_path).read_text()
    team_max_hp = int(_re.search(r'TEAM_MAX_HP\s*=\s*(\d+)', text).group(1))
    enemy_max_hp = int(_re.search(r'ENEMY_MAX_HP\s*=\s*(\d+)', text).group(1))
    boss_damage = float(_re.search(r'BOSS_DAMAGE\s*=\s*([\d.]+)', text).group(1))
    ps_match = _re.search(r'PLAYER_STATS\s*=\s*(?:jnp\.array|np\.array)?\(?\s*(\[\[.+?\]\])\s*\)?', text, _re.DOTALL)
    rows = _re.findall(r'\[([^\[\]]+)\]', ps_match.group(1))
    player_stats = np.array([[float(x) for x in row.split(',')] for row in rows])
    class Config: pass
    cfg = Config()
    cfg.TEAM_MAX_HP = team_max_hp
    cfg.ENEMY_MAX_HP = enemy_max_hp
    cfg.BOSS_DAMAGE = boss_damage
    cfg.PLAYER_STATS = player_stats
    return cfg

oms.load_config_module = _load_config_no_jax

# Load data
DATA_DIRS = [
    str(EXPORTS_DIR / 'bayesian-role-specialization-2026-03-06-09-54-19'),
    str(EXPORTS_DIR / 'bayesian-role-specialization-2026-03-18-15-47-09'),
]

records = oms.load_team_rounds(data_dirs=DATA_DIRS)
n_envs = len(set(r['env_id'] for r in records))
print(f'Loaded {len(records)} team-rounds across {n_envs} environments')

Loaded 66 team-rounds across 6 environments


## 2. Aggregate Fine-Tuning

In [3]:
metric = 'combo_r'

def pick_best(results, metric='combo_r'):
    valid = [r for r in results if not np.isnan(r.get(metric, float('nan')))]
    if not valid:
        return results[0]
    return max(valid, key=lambda r: r[metric])

# Grid ranges
tau_prior_min,   tau_prior_max,   tau_prior_steps   = 0.1, 20.0, 10
tau_softmax_min, tau_softmax_max, tau_softmax_steps = 0.1, 20.0, 10
eps_min, eps_max, eps_steps = 0.001, 0.2, 10

tau_prior_vals   = np.linspace(tau_prior_min, tau_prior_max, tau_prior_steps)
tau_softmax_vals = np.linspace(tau_softmax_min, tau_softmax_max, tau_softmax_steps)
eps_vals         = np.geomspace(eps_min, eps_max, eps_steps)

tp_step  = (tau_prior_max - tau_prior_min) / tau_prior_steps
ts_step  = (tau_softmax_max - tau_softmax_min) / tau_softmax_steps
eps_ratio = (eps_max / eps_min) ** (1.0 / eps_steps)

def evaluate_bv(records, tau_prior, tau_softmax, epsilon):
    """Run Bayesian-Value and return fit metrics."""
    results = oms.run_all_predictions(records, tau_prior=tau_prior,
                                      tau_softmax=tau_softmax, epsilon=epsilon)
    corrs = compute_pearson(results)
    ll = compute_log_likelihood(results)
    g = corrs.get('__global__', {})
    return {
        'tau_prior': tau_prior, 'tau_softmax': tau_softmax, 'epsilon': epsilon,
        'combo_r': g.get('combo', {}).get('r', float('nan')),
        'marg_r': g.get('marginal', {}).get('r', float('nan')),
        'mean_ll': float(np.mean([v['mean_ll'] for v in ll.values()])) if ll else float('nan'),
    }

# Coarse grid (same ranges, 3 params)
bv_sweep = []
bv_combos = list(product(tau_prior_vals, tau_softmax_vals, eps_vals))
print(f'Bayesian-Value coarse grid: {len(tau_prior_vals)} x {len(tau_softmax_vals)} x {len(eps_vals)} = {len(bv_combos)} points')
for i, (tp, ts, eps) in enumerate(bv_combos, 1):
    if i % 100 == 0 or i == len(bv_combos):
        print(f'  [{i}/{len(bv_combos)}] ...', flush=True)
    bv_sweep.append(evaluate_bv(records, tp, ts, eps))

best_bv = pick_best(bv_sweep, metric)
print(f'\nCoarse best: tau_prior={best_bv["tau_prior"]:.4f} tau_softmax={best_bv["tau_softmax"]:.4f} '
      f'eps={best_bv["epsilon"]:.6f}')
print(f'  combo_r={best_bv["combo_r"]:.4f}  marg_r={best_bv["marg_r"]:.4f}  mean_ll={best_bv["mean_ll"]:.4f}')

Bayesian-Value coarse grid: 10 x 10 x 10 = 1000 points
  [100/1000] ...
  [200/1000] ...
  [300/1000] ...
  [400/1000] ...
  [500/1000] ...
  [600/1000] ...
  [700/1000] ...
  [800/1000] ...
  [900/1000] ...
  [1000/1000] ...

Coarse best: tau_prior=15.5778 tau_softmax=4.5222 eps=0.200000
  combo_r=0.4009  marg_r=0.4546  mean_ll=-3.8982


### Step 2: Refined Grid

In [4]:
# Refined grid for Bayesian-Value
bv_fine_tp = np.linspace(
    max(tau_prior_min, best_bv['tau_prior'] - tp_step),
    min(tau_prior_max, best_bv['tau_prior'] + tp_step), 10)
bv_fine_ts = np.linspace(
    max(tau_softmax_min, best_bv['tau_softmax'] - ts_step),
    min(tau_softmax_max, best_bv['tau_softmax'] + ts_step), 10)
bv_fine_eps = np.geomspace(
    max(eps_min, best_bv['epsilon'] / eps_ratio),
    min(eps_max, best_bv['epsilon'] * eps_ratio), 10)

bv_fine_total = len(bv_fine_tp) * len(bv_fine_ts) * len(bv_fine_eps)
print(f'Refined grid: {bv_fine_total} points')
for i, (tp, ts, eps) in enumerate(product(bv_fine_tp, bv_fine_ts, bv_fine_eps), 1):
    if i % 100 == 0 or i == bv_fine_total:
        print(f'  [{i}/{bv_fine_total}] ...', flush=True)
    bv_sweep.append(evaluate_bv(records, tp, ts, eps))

best_bv = pick_best(bv_sweep, metric)
print(f'\nRefined best: tau_prior={best_bv["tau_prior"]:.4f} tau_softmax={best_bv["tau_softmax"]:.4f} '
      f'eps={best_bv["epsilon"]:.6f}')
print(f'  combo_r={best_bv["combo_r"]:.4f}  marg_r={best_bv["marg_r"]:.4f}  mean_ll={best_bv["mean_ll"]:.4f}')

Refined grid: 1000 points
  [100/1000] ...
  [200/1000] ...
  [300/1000] ...
  [400/1000] ...
  [500/1000] ...
  [600/1000] ...
  [700/1000] ...
  [800/1000] ...
  [900/1000] ...
  [1000/1000] ...

Refined best: tau_prior=17.5678 tau_softmax=3.4167 eps=0.200000
  combo_r=0.4022  marg_r=0.4679  mean_ll=-4.4887


### Step 3: Scipy Polishing

In [5]:
# Scipy polishing for Bayesian-Value
def objective_bv(params):
    tp, ts, eps = params
    res = evaluate_bv(records, tp, ts, eps)
    return -res[metric]

bv_tp_lo = max(tau_prior_min, best_bv['tau_prior'] - tp_step / 2)
bv_tp_hi = min(tau_prior_max, best_bv['tau_prior'] + tp_step / 2)
bv_ts_lo = max(tau_softmax_min, best_bv['tau_softmax'] - ts_step / 2)
bv_ts_hi = min(tau_softmax_max, best_bv['tau_softmax'] + ts_step / 2)
bv_eps_lo = max(1e-6, best_bv['epsilon'] / 2)
bv_eps_hi = min(eps_max, best_bv['epsilon'] * 2)

bv_x0 = [best_bv['tau_prior'], best_bv['tau_softmax'], best_bv['epsilon']]
bv_bounds = [(bv_tp_lo, bv_tp_hi), (bv_ts_lo, bv_ts_hi), (bv_eps_lo, bv_eps_hi)]

print('Scipy optimization (Bayesian-Value) ...')
bv_opt = minimize(objective_bv, bv_x0, method='L-BFGS-B', bounds=bv_bounds,
                  options={'maxiter': 50, 'ftol': 1e-6})
bv_opt_tp, bv_opt_ts, bv_opt_eps = bv_opt.x
print(f'  Optimal: tau_prior={bv_opt_tp:.4f} tau_softmax={bv_opt_ts:.4f} eps={bv_opt_eps:.6f}')

bv_opt_eval = evaluate_bv(records, bv_opt_tp, bv_opt_ts, bv_opt_eps)
bv_sweep.append(bv_opt_eval)
best_bv = pick_best(bv_sweep, metric)
print(f'\nFinal best BV: tau_prior={best_bv["tau_prior"]:.4f} tau_softmax={best_bv["tau_softmax"]:.4f} '
      f'eps={best_bv["epsilon"]:.6f}')
print(f'  combo_r={best_bv["combo_r"]:.4f}  marg_r={best_bv["marg_r"]:.4f}  mean_ll={best_bv["mean_ll"]:.4f}')

Scipy optimization (Bayesian-Value) ...
  Optimal: tau_prior=17.5678 tau_softmax=3.4170 eps=0.200000

Final best BV: tau_prior=17.5678 tau_softmax=3.4170 eps=0.200000
  combo_r=0.4022  marg_r=0.4679  mean_ll=-4.4884


## 3. Switch-Stage Fine-Tuning

In [6]:
def filter_switch_stages(all_results):
    """Keep only predictions where the human combo changed from the previous stage."""
    filtered = {}
    for env_id, data in all_results.items():
        new_team_preds = []
        for team_preds in data['team_predictions']:
            filtered_preds = []
            for s, pred in enumerate(team_preds):
                if s > 0 and pred['human_combo'] != team_preds[s - 1]['human_combo']:
                    filtered_preds.append(pred)
            new_team_preds.append(filtered_preds)

        canon_combos = data['canonical_combos']
        stat_profile = data['stat_profile']
        stage_predicted = defaultdict(lambda: defaultdict(float))
        stage_human = defaultdict(lambda: defaultdict(int))
        stage_model_marg = defaultdict(lambda: np.zeros(3))
        stage_human_marg = defaultdict(lambda: np.zeros(3))
        stage_counts = defaultdict(int)
        max_stages = 0

        for team_preds in new_team_preds:
            for s, pred in enumerate(team_preds):
                stage_counts[s] += 1
                max_stages = max(max_stages, s + 1)
                for combo, prob in pred['predicted_dist'].items():
                    stage_predicted[s][canonical_combo(combo, stat_profile)] += prob
                stage_human[s][canonical_combo(pred['human_combo'], stat_profile)] += 1
                stage_model_marg[s] += pred['model_marginal']
                stage_human_marg[s] += combo_marginal(pred['human_combo'])

        if max_stages == 0:
            continue

        predicted_avg, model_marg_avg, human_marg_avg = {}, {}, {}
        for s in range(max_stages):
            n = stage_counts[s]
            if n > 0:
                predicted_avg[s] = {cc: stage_predicted[s].get(cc, 0.0) / n for cc in canon_combos}
                model_marg_avg[s] = stage_model_marg[s] / n
                human_marg_avg[s] = stage_human_marg[s] / n

        filtered[env_id] = dict(data)
        filtered[env_id].update({
            'max_stages': max_stages,
            'stage_predicted': predicted_avg,
            'stage_human': dict(stage_human),
            'stage_counts': dict(stage_counts),
            'team_predictions': new_team_preds,
            'stage_model_marginal': model_marg_avg,
            'stage_human_marginal': human_marg_avg,
        })

    return filtered

In [7]:
def evaluate_bv_switch(records, tau_prior, tau_softmax, epsilon):
    """Run BV on full history, compute metrics only on switch stages."""
    results = oms.run_all_predictions(records, tau_prior=tau_prior,
                                      tau_softmax=tau_softmax, epsilon=epsilon)
    sw_results = filter_switch_stages(results)
    if not sw_results:
        return {'tau_prior': tau_prior, 'tau_softmax': tau_softmax, 'epsilon': epsilon,
                'combo_r': float('nan'), 'marg_r': float('nan'), 'mean_ll': float('nan')}
    corrs = compute_pearson(sw_results)
    ll = compute_log_likelihood(sw_results)
    g = corrs.get('__global__', {})
    return {
        'tau_prior': tau_prior, 'tau_softmax': tau_softmax, 'epsilon': epsilon,
        'combo_r': g.get('combo', {}).get('r', float('nan')),
        'marg_r': g.get('marginal', {}).get('r', float('nan')),
        'mean_ll': float(np.mean([v['mean_ll'] for v in ll.values()])) if ll else float('nan'),
    }

# Coarse grid
bv_sw_sweep = []
bv_sw_combos = list(product(tau_prior_vals, tau_softmax_vals, eps_vals))
print(f'BV switch-stage coarse grid: {len(bv_sw_combos)} points')
for i, (tp, ts, eps) in enumerate(bv_sw_combos, 1):
    if i % 100 == 0 or i == len(bv_sw_combos):
        print(f'  [{i}/{len(bv_sw_combos)}] ...', flush=True)
    bv_sw_sweep.append(evaluate_bv_switch(records, tp, ts, eps))

best_bv_sw = pick_best(bv_sw_sweep, metric)
print(f'\nCoarse best: tau_prior={best_bv_sw["tau_prior"]:.4f} tau_softmax={best_bv_sw["tau_softmax"]:.4f} '
      f'eps={best_bv_sw["epsilon"]:.6f}')
print(f'  combo_r={best_bv_sw["combo_r"]:.4f}  marg_r={best_bv_sw["marg_r"]:.4f}')

BV switch-stage coarse grid: 1000 points
  [100/1000] ...
  [200/1000] ...
  [300/1000] ...
  [400/1000] ...
  [500/1000] ...
  [600/1000] ...
  [700/1000] ...
  [800/1000] ...
  [900/1000] ...
  [1000/1000] ...

Coarse best: tau_prior=0.1000 tau_softmax=0.1000 eps=0.061616
  combo_r=0.2865  marg_r=0.3873


In [8]:
# Refined grid for BV switch-stage
bv_sw_fine_tp = np.linspace(
    max(tau_prior_min, best_bv_sw['tau_prior'] - tp_step),
    min(tau_prior_max, best_bv_sw['tau_prior'] + tp_step), 10)
bv_sw_fine_ts = np.linspace(
    max(tau_softmax_min, best_bv_sw['tau_softmax'] - ts_step),
    min(tau_softmax_max, best_bv_sw['tau_softmax'] + ts_step), 10)
bv_sw_fine_eps = np.geomspace(
    max(eps_min, best_bv_sw['epsilon'] / eps_ratio),
    min(eps_max, best_bv_sw['epsilon'] * eps_ratio), 10)

bv_sw_fine_total = len(bv_sw_fine_tp) * len(bv_sw_fine_ts) * len(bv_sw_fine_eps)
print(f'Refined grid: {bv_sw_fine_total} points')
for i, (tp, ts, eps) in enumerate(product(bv_sw_fine_tp, bv_sw_fine_ts, bv_sw_fine_eps), 1):
    if i % 100 == 0 or i == bv_sw_fine_total:
        print(f'  [{i}/{bv_sw_fine_total}] ...', flush=True)
    bv_sw_sweep.append(evaluate_bv_switch(records, tp, ts, eps))

best_bv_sw = pick_best(bv_sw_sweep, metric)

# Scipy polish
def objective_bv_sw(params):
    return -evaluate_bv_switch(records, *params)[metric]

bv_sw_opt = minimize(objective_bv_sw,
    [best_bv_sw['tau_prior'], best_bv_sw['tau_softmax'], best_bv_sw['epsilon']],
    method='L-BFGS-B',
    bounds=[(max(tau_prior_min, best_bv_sw['tau_prior'] - tp_step/2),
             min(tau_prior_max, best_bv_sw['tau_prior'] + tp_step/2)),
            (max(tau_softmax_min, best_bv_sw['tau_softmax'] - ts_step/2),
             min(tau_softmax_max, best_bv_sw['tau_softmax'] + ts_step/2)),
            (max(1e-6, best_bv_sw['epsilon'] / 2),
             min(eps_max, best_bv_sw['epsilon'] * 2))],
    options={'maxiter': 50, 'ftol': 1e-6})
bv_sw_sweep.append(evaluate_bv_switch(records, *bv_sw_opt.x))
best_bv_sw = pick_best(bv_sw_sweep, metric)

print(f'\nFinal best BV (switch-stage): tau_prior={best_bv_sw["tau_prior"]:.4f} '
      f'tau_softmax={best_bv_sw["tau_softmax"]:.4f} eps={best_bv_sw["epsilon"]:.6f}')
print(f'  combo_r={best_bv_sw["combo_r"]:.4f}  marg_r={best_bv_sw["marg_r"]:.4f}')

Refined grid: 1000 points
  [100/1000] ...
  [200/1000] ...
  [300/1000] ...
  [400/1000] ...
  [500/1000] ...
  [600/1000] ...
  [700/1000] ...
  [800/1000] ...
  [900/1000] ...
  [1000/1000] ...

Final best BV (switch-stage): tau_prior=0.1000 tau_softmax=0.1000 eps=0.059586
  combo_r=0.2865  marg_r=0.3873


## 4. Save Parameters

In [9]:
output = {
    'metric_optimized': metric,
    'aggregate_tuned': {
        'tau_prior': best_bv['tau_prior'], 'tau_softmax': best_bv['tau_softmax'],
        'epsilon': best_bv['epsilon'],
        'combo_r': best_bv['combo_r'], 'marg_r': best_bv['marg_r'], 'mean_ll': best_bv['mean_ll'],
    },
    'switch_stage_tuned': {
        'tau_prior': best_bv_sw['tau_prior'], 'tau_softmax': best_bv_sw['tau_softmax'],
        'epsilon': best_bv_sw['epsilon'],
        'combo_r': best_bv_sw['combo_r'], 'marg_r': best_bv_sw['marg_r'], 'mean_ll': best_bv_sw['mean_ll'],
    },
}

with open('bayesian_value_params.json', 'w') as f:
    json.dump(output, f, indent=2)
print('Saved to bayesian_value_params.json')
print(json.dumps(output, indent=2))

Saved to bayesian_value_params.json
{
  "metric_optimized": "combo_r",
  "aggregate_tuned": {
    "tau_prior": 17.567816724398256,
    "tau_softmax": 3.4170335232140374,
    "epsilon": 0.2,
    "combo_r": 0.4021983136662147,
    "marg_r": 0.4679279807942096,
    "mean_ll": -4.4884377178285435
  },
  "switch_stage_tuned": {
    "tau_prior": 0.1,
    "tau_softmax": 0.1,
    "epsilon": 0.05958645275895692,
    "combo_r": 0.28653863613441627,
    "marg_r": 0.387337780326084,
    "mean_ll": -25.83433200775544
  }
}
